In [1]:
# 1. Load Data
# 2. Prepare Features and Target
# 3. Train/Test Split
# 4. Preprocessing Pipeline
# 5. Logistic Regression Baseline
# 6. Random Forest
# 7. XGBoost
# 8. Model Comparison
# 9. Feature Importance / SHAP

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
df = pd.read_csv("../data/raw/device_telemetry.csv")

print(df.shape)
df.head()

(12020, 19)


,Device_ID,OS,OS_Version,Manufacturer,Model,Network_Type,Battery_Health,Battery_Temperature,CPU_Usage,Memory_Usage,Storage_Usage,Network_Signal,App_Crash_Count,Reboot_Count,Error_Count,Data_Usage,Days_Since_Last_Update,Last_Checkin_Hours,Failure_Within_7_Days
0,DEV-00001,Android,13.0,Apple,Model_C,4G,83.264927,39.553141,75.992968,53.706658,71.905273,60.879876,2,4,7,3141.410848,3,1.693449,0
1,DEV-00002,iOS,11.0,Dell,Model_D,WiFi,86.814058,33.264014,79.095793,58.568440,60.156997,57.391034,2,2,1,1610.601317,91,29.440840,0
2,DEV-00003,Windows,12.0,Samsung,Model_E,WiFi,88.492257,40.148116,59.021509,56.214448,48.434434,33.145753,0,1,2,2584.005107,86,2.555599,0
3,DEV-00004,Windows,14.0,Honeywell,Model_D,WiFi,73.924453,36.292322,49.454212,53.309786,75.514666,99.447988,3,1,2,1273.264021,47,3.815414,0
4,DEV-00005,Android,13.0,Honeywell,Model_E,WiFi,83.267405,43.406349,20.828683,48.839864,94.537607,67.980414,1,1,3,1724.513745,169,7.058946,0


In [4]:
X = df.drop(columns=["Failure_Within_7_Days", "Device_ID"])
y = df["Failure_Within_7_Days"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [6]:
categorical_features = [
    "OS",
    "OS_Version",
    "Manufacturer",
    "Model",
    "Network_Type"
]

numerical_features = [
    "Battery_Health",
    "Battery_Temperature",
    "CPU_Usage",
    "Memory_Usage",
    "Storage_Usage",
    "Network_Signal",
    "App_Crash_Count",
    "Reboot_Count",
    "Error_Count",
    "Data_Usage",
    "Days_Since_Last_Update",
    "Last_Checkin_Hours"
]

In [7]:
from sklearn.preprocessing import StandardScaler

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [8]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor.fit(X_train)

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [9]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [10]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(9616, 37)
(2404, 37)


In [11]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [12]:
logistic_model.fit(
    X_train_processed,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [13]:
y_pred = logistic_model.predict(X_test_processed)

y_pred_proba = logistic_model.predict_proba(
    X_test_processed
)[:, 1]

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("Accuracy :", accuracy) #percentage of correctly predicted samples
print("Precision:", precision) #percentage of correctly predicted positive samples (correctly predicted positive samples / total predicted positive samples)
print("Recall   :", recall) #percentage of actual positive samples correctly predicted (correctly predicted positive samples / total actual positive samples)
print("F1 Score :", f1) #harmonic mean of precision and recall
print("ROC-AUC  :", roc_auc) #area under the receiver operating characteristic curve

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

Accuracy : 0.8248752079866889
Precision: 0.7396226415094339
Recall   : 0.5807407407407408
F1 Score : 0.6506224066390042
ROC-AUC  : 0.8649512670565301

Confusion Matrix:
[[1591  138]
 [ 283  392]]


In [15]:
# Recall is less -> chance of predicting failing systems is less
# Now let's train a Random Forest as our second model to capture non linear relationship

In [16]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

In [17]:
random_forest_model.fit(
    X_train_processed,
    y_train
)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
y_pred_rf = random_forest_model.predict(
    X_test_processed
)

y_pred_proba_rf = random_forest_model.predict_proba(
    X_test_processed
)[:, 1]

In [19]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)
print("ROC-AUC  :", roc_auc_rf)

cm_rf = confusion_matrix(y_test, y_pred_rf)

print("\nConfusion Matrix:")
print(cm_rf)

Accuracy : 0.7982529118136439
Precision: 0.7668539325842697
Recall   : 0.40444444444444444
F1 Score : 0.5295829291949563
ROC-AUC  : 0.8386714649872544

Confusion Matrix:
[[1646   83]
 [ 402  273]]


In [ ]:
#Tuning the classification threshold for the random forest model based on precision, recall, and F1 score

In [20]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

for threshold in thresholds:

    y_pred_threshold = (
        y_pred_proba_rf >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold
    )

    recall = recall_score(
        y_test,
        y_pred_threshold
    )

    f1 = f1_score(
        y_test,
        y_pred_threshold
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.50 | Precision: 0.758 | Recall: 0.413 | F1: 0.535
Threshold: 0.45 | Precision: 0.715 | Recall: 0.524 | F1: 0.605
Threshold: 0.40 | Precision: 0.652 | Recall: 0.624 | F1: 0.637
Threshold: 0.35 | Precision: 0.590 | Recall: 0.705 | F1: 0.642
Threshold: 0.30 | Precision: 0.529 | Recall: 0.769 | F1: 0.627


Logistic Regression achieved ROC-AUC 0.865 and recall 58.1% at the default threshold. Random Forest achieved ROC-AUC 0.839, while threshold analysis showed that lowering the Random Forest threshold to 0.35 increased recall to 70.5%, with precision of 59.0%.


In [21]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=15,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train_processed, y_train)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,15
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [22]:
print("Best parameters:")
print(random_search.best_params_)

print("\nBest CV F1:")
print(random_search.best_score_)

Best parameters:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}

Best CV F1:
0.5380000540064821


In [23]:
tuned_rf = random_search.best_estimator_

In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

y_pred_tuned = tuned_rf.predict(X_test_processed)

y_pred_proba_tuned = tuned_rf.predict_proba(
    X_test_processed
)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall   :", recall_score(y_test, y_pred_tuned))
print("F1 Score :", f1_score(y_test, y_pred_tuned))
print("ROC-AUC  :", roc_auc_score(y_test, y_pred_proba_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Accuracy : 0.7961730449251248
Precision: 0.7650429799426934
Recall   : 0.39555555555555555
F1 Score : 0.521484375
ROC-AUC  : 0.838102949681897

Confusion Matrix:
[[1647   82]
 [ 408  267]]


In [27]:
best_params = random_search.best_params_

print(best_params)

{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}


In [25]:
thresholds = [0.50, 0.45, 0.40, 0.35, 0.30]

for threshold in thresholds:

    y_pred_threshold = (
        y_pred_proba_tuned >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold
    )

    recall = recall_score(
        y_test,
        y_pred_threshold
    )

    f1 = f1_score(
        y_test,
        y_pred_threshold
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.50 | Precision: 0.765 | Recall: 0.396 | F1: 0.521
Threshold: 0.45 | Precision: 0.709 | Recall: 0.502 | F1: 0.588
Threshold: 0.40 | Precision: 0.650 | Recall: 0.599 | F1: 0.623
Threshold: 0.35 | Precision: 0.588 | Recall: 0.693 | F1: 0.636
Threshold: 0.30 | Precision: 0.536 | Recall: 0.776 | F1: 0.634


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib

final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        **best_params,
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [29]:
joblib.dump(
    final_model,
    "../models/device_failure_model.joblib"
)

joblib.dump(
    0.35,
    "../models/failure_threshold.joblib"
)

print("Final model and threshold saved successfully!")

Final model and threshold saved successfully!
